# 🏍️ SMART HELMET DETECTION SYSTEM (HỆ THỐNG PHÁT HIỆN MŨ BẢO HIỂM)
### Đề tài Đồ án - Chuyên ngành Hệ thống Thông tin (Information Systems)
**Mô hình:** YOLO11 Nano (`yolo11n.pt`) | **Phần cứng:** Google Colab GPU (Tesla T4)

---
Notebook này hướng dẫn quy trình huấn luyện toàn diện mô hình nhận diện mũ bảo hiểm trên Google Colab:
1. **Cấu hình môi trường GPU miễn phí trên Colab**
2. **Tải lên & Giải nén bộ dữ liệu sạch từ máy tính (`curated_v1_colab.zip`)**
3. **Huấn luyện mô hình YOLO11n (epochs=80, imgsz=416, batch=16)**
4. **Đánh giá các chỉ số (mAP50, Precision, Recall, Confusion Matrix)**
5. **Thử nghiệm suy luận trên Ảnh & Video với ID độc lập và màu Đỏ/Xanh**
6. **Tải file trọng số `best.pt` về máy tính để tích hợp vào hệ thống**

## Bước 1: Kiểm tra phần cứng GPU của Google Colab
> Lưu ý: Vào menu **Runtime** -> **Change runtime type** -> chọn **T4 GPU** trước khi chạy!

In [ ]:
!nvidia-smi

## Bước 2: Cài đặt thư viện Ultralytics và các gói bổ trợ

In [ ]:
!pip install --upgrade ultralytics opencv-python pyyaml matplotlib

## Bước 3: Tải bộ dữ liệu từ máy tính lên Google Colab (`curated_v1_colab.zip`)

Chạy ô code bên dưới. Hệ thống sẽ mở nút **'Choose Files'** để bạn chọn file `curated_v1_colab.zip` từ máy tính (nằm trong thư mục dự án `d:\IOT\curated_v1_colab.zip`).
*Lưu ý: Nếu bạn đã kéo thả file zip vào thanh tệp của Colab trước đó, code sẽ tự động phát hiện và giải nén ngay mà không cần tải lại.*

In [ ]:
import os
import zipfile
from google.colab import files

dataset_dir = "/content/dataset"
os.makedirs(dataset_dir, exist_ok=True)
zip_path = "/content/curated_v1_colab.zip"

# Kiểm tra nếu chưa có file zip trên Colab, mở cửa sổ upload từ máy tính
if not os.path.exists(zip_path):
    print("📁 Vui lòng bấm nút 'Choose Files' bên dưới để chọn file 'curated_v1_colab.zip' từ máy tính:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".zip"):
            zip_path = os.path.join("/content", filename)
            break

if os.path.exists(zip_path):
    print(f"📦 Đang giải nén bộ dữ liệu từ {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print(f"✅ Giải nén thành công toàn bộ dữ liệu vào: {dataset_dir}")
else:
    print("❌ Chưa tìm thấy file zip dữ liệu. Vui lòng chạy lại ô này để chọn tải file lên!")

## Bước 4: Kiểm tra cấu hình `data.yaml`

In [ ]:
import yaml

yaml_path = "/content/dataset/data.yaml"
if os.path.exists(yaml_path):
    with open(yaml_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    print("Cấu hình Dataset:")
    print(yaml.dump(config, sort_keys=False))
else:
    # Tự động tạo nếu chưa có
    config = {
        "path": "/content/dataset",
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": 2,
        "names": {0: "With Helmet", 1: "Without Helmet"}
    }
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)
    print("Đã khởi tạo data.yaml mới tại", yaml_path)

## Bước 5: Bắt đầu Huấn luyện mô hình YOLO11n trên GPU

* **Kiến trúc:** YOLO11 nano (`yolo11n.pt`)
* **Resolution:** 416x416 px (Tối ưu cho Edge/IoT)
* **Batch size:** 16 (Tận dụng GPU 16GB của Colab)
* **Epochs:** 80 (Early stopping với patience = 20)

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình YOLO11n với trọng số pretrained từ COCO
model = YOLO("yolo11n.pt")

# Tiến hành huấn luyện
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=80,
    imgsz=416,
    batch=16,
    patience=20,
    device=0,
    workers=2,
    seed=42,
    project="smart_helmet_colab",
    name="yolo11n_run",
    save=True,
    plots=True,
    exist_ok=True
)

print("\n✅ QUÁ TRÌNH HUẤN LUYỆN ĐÃ HOÀN THÀNH!")

## Bước 6: Trực quan hóa kết quả huấn luyện (Charts & Metrics)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = "/content/smart_helmet_colab/yolo11n_run"

# Hiển thị biểu đồ quá trình học (loss, mAP)
results_img = os.path.join(run_dir, "results.png")
if os.path.exists(results_img):
    plt.figure(figsize=(16, 10))
    img = mpimg.imread(results_img)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Biểu đồ huấn luyện & Đánh giá (Training & Validation Metrics)", fontsize=14)
    plt.show()

# Hiển thị Confusion Matrix
cm_img = os.path.join(run_dir, "confusion_matrix_normalized.png")
if os.path.exists(cm_img):
    plt.figure(figsize=(8, 8))
    img = mpimg.imread(cm_img)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix Chuẩn hóa", fontsize=14)
    plt.show()

## Bước 7: Đánh giá mô hình tốt nhất (`best.pt`) trên tập Validation & Test

In [ ]:
import os
import glob
from ultralytics import YOLO

# 1. Tự động xác định đường dẫn best.pt mới nhất
best_weights = None
if 'run_dir' in locals() and run_dir and os.path.exists(os.path.join(run_dir, "weights", "best.pt")):
    best_weights = os.path.join(run_dir, "weights", "best.pt")
else:
    candidates = glob.glob("/content/**/weights/best.pt", recursive=True)
    if candidates:
        candidates.sort(key=os.path.getmtime, reverse=True)
        best_weights = candidates[0]

# 2. Tự động tìm file data.yaml
data_yaml = "/content/dataset/data.yaml"
if not os.path.exists(data_yaml):
    yaml_candidates = glob.glob("/content/**/data.yaml", recursive=True)
    if yaml_candidates:
        data_yaml = yaml_candidates[0]

if best_weights and os.path.exists(best_weights):
    print(f"🎯 Đang nạp mô hình từ: {best_weights}")
    trained_model = YOLO(best_weights)
    
    print("\n⏳ Đang tiến hành đánh giá trên tập Validation...")
    val_metrics = trained_model.val(data=data_yaml, split="val", imgsz=416, verbose=True)
    
    print("\n" + "="*50)
    print("📊 KẾT QUẢ ĐÁNH GIÁ TỔNG THỂ (VALIDATION METRICS)")
    print("="*50)
    print(f"🔹 Precision (Độ chính xác)  : {val_metrics.box.mp * 100:.2f}%")
    print(f"🔹 Recall (Độ phủ/thu hồi)   : {val_metrics.box.mr * 100:.2f}%")
    print(f"🔹 mAP@50 (Độ chuẩn xác 50%) : {val_metrics.box.map50 * 100:.2f}%")
    print(f"🔹 mAP@50-95 (mAP chuẩn hóa) : {val_metrics.box.map * 100:.2f}%")
    print("="*50)
else:
    print("❌ Chưa tìm thấy file trọng số best.pt. Hãy đảm bảo Bước 5 đã hoàn thành!")

## Bước 8: Thử nghiệm suy luận với ID độc lập và quy chuẩn màu sắc
* 🟢 **Xanh lá cây (Green):** `With Helmet`
* 🔴 **Đỏ tươi (Red):** `Without Helmet`
* **ID độc lập** hiển thị trên từng Bounding Box

In [ ]:
import cv2
import os
import glob
from pathlib import Path
from PIL import Image
from IPython.display import display

# Bảng màu quy chuẩn (BGR cho OpenCV)
COLORS = {
    0: (0, 200, 0),      # With Helmet -> Xanh lá cây (Green)
    1: (0, 0, 255),      # Without Helmet -> Đỏ tươi (Red)
}
CLASS_NAMES = {0: "With Helmet", 1: "Without Helmet"}

# Đảm bảo đã nạp model
if 'trained_model' not in locals():
    candidates = glob.glob("/content/**/weights/best.pt", recursive=True)
    if candidates:
        candidates.sort(key=os.path.getmtime, reverse=True)
        from ultralytics import YOLO
        trained_model = YOLO(candidates[0])

# Tìm ảnh trong tập test (hỗ trợ mọi định dạng: jpg, png, jpeg)
test_dir = Path("/content/dataset/test/images")
test_images = [str(p) for p in test_dir.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]

if test_images and 'trained_model' in locals():
    # Thử nghiệm trên 2 ảnh đầu tiên của tập test
    samples = test_images[:2]
    for idx_img, sample_img_path in enumerate(samples):
        print(f"\n📸 Ảnh kiểm thử {idx_img + 1}: {os.path.basename(sample_img_path)}")
        preds = trained_model.predict(sample_img_path, conf=0.35, imgsz=416, verbose=False)[0]
        annotated_bgr = cv2.imread(sample_img_path)
        h_img, w_img = annotated_bgr.shape[:2]
        
        # Vẽ từng bounding box với ID độc lập
        if preds.boxes is not None and len(preds.boxes) > 0:
            for idx_box, box in enumerate(preds.boxes):
                x1, y1, x2, y2 = [int(round(v)) for v in box.xyxy[0].tolist()]
                cls_id = int(box.cls[0].item())
                conf = float(box.conf[0].item())
                color = COLORS.get(cls_id, (255, 255, 255))
                label = f"ID:{idx_box+1} | {CLASS_NAMES.get(cls_id, str(cls_id))} {conf:.2f}"
                
                # Vẽ khung hình chữ nhật
                cv2.rectangle(annotated_bgr, (x1, y1), (x2, y2), color, 2)
                # Vẽ nền nhãn để chữ rõ nét
                (text_w, text_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
                top = max(0, y1 - text_h - baseline - 4)
                cv2.rectangle(annotated_bgr, (x1, top), (min(w_img - 1, x1 + text_w + 4), top + text_h + baseline + 4), color, -1)
                cv2.putText(annotated_bgr, label, (x1 + 2, top + text_h + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
        else:
            print("  (Không phát hiện đối tượng nào)")
            
        # Vẽ bảng chú thích Legend ở góc dưới bên trái
        legend_items = [(0, "With Helmet (Green)"), (1, "Without Helmet (Red)")]
        for row_i, (cid, cname) in enumerate(legend_items):
            l_bottom = h_img - 8 - row_i * 20
            cv2.rectangle(annotated_bgr, (6, l_bottom - 16), (210, l_bottom), COLORS[cid], -1)
            cv2.putText(annotated_bgr, cname, (10, l_bottom - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)

        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        display(Image.fromarray(annotated_rgb))
else:
    print("❌ Không tìm thấy ảnh trong /content/dataset/test/images hoặc chưa nạp được trained_model!")

In [ ]:
import cv2
import os
from pathlib import Path

demo_frames_dir = Path("/content/dataset/test/images")
demo_images = sorted([str(p) for p in demo_frames_dir.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])[:15]

if demo_images and 'trained_model' in locals():
    out_video_path = "/content/test_video_result.mp4"
    print(f"🎬 Đang tạo video kiểm thử từ {len(demo_images)} ảnh tập Test...")
    
    first_frame = cv2.imread(demo_images[0])
    h, w = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_video_path, fourcc, 2.0, (w, h))
    
    for img_p in demo_images:
        frame = cv2.imread(img_p)
        preds = trained_model.predict(frame, conf=0.35, imgsz=416, verbose=False)[0]
        if preds.boxes is not None:
            for idx, box in enumerate(preds.boxes):
                x1, y1, x2, y2 = [int(round(v)) for v in box.xyxy[0].tolist()]
                cls_id = int(box.cls[0].item())
                conf = float(box.conf[0].item())
                color = COLORS.get(cls_id, (255, 255, 255))
                label = f"ID:{idx+1} | {CLASS_NAMES.get(cls_id, str(cls_id))} {conf:.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
                top = max(0, y1 - th - bl - 4)
                cv2.rectangle(frame, (x1, top), (min(w - 1, x1 + tw + 4), top + th + bl + 4), color, -1)
                cv2.putText(frame, label, (x1 + 2, top + th + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
        
        # Mỗi ảnh ghi 2 lần để hiển thị trong 1 giây ở 2 FPS
        writer.write(frame)
        writer.write(frame)
        
    writer.release()
    print(f"✅ Đã xuất video kiểm thử thành công tại: {out_video_path}")
    print("💡 Bounding box có ID độc lập, With Helmet màu Xanh và Without Helmet màu Đỏ!")
else:
    print("⚠️ Chưa tìm thấy ảnh test hoặc model để tạo video.")

## Bước 9: Tải file trọng số `best.pt` về máy tính
> Chạy cell bên dưới để trình duyệt tự động tải file `best.pt` về máy tính.
> Sau đó, copy file này vào thư mục `weights/best.pt` của dự án trên máy tính để chạy Video Inference!

In [ ]:
from google.colab import files
import os
import glob

# 1. Tự động tìm file best.pt mới nhất
target_pt = None
if 'best_weights' in locals() and best_weights and os.path.exists(best_weights):
    target_pt = best_weights
elif 'run_dir' in locals() and run_dir and os.path.exists(os.path.join(run_dir, "weights", "best.pt")):
    target_pt = os.path.join(run_dir, "weights", "best.pt")
else:
    candidates = glob.glob("/content/**/weights/best.pt", recursive=True)
    if candidates:
        candidates.sort(key=os.path.getmtime, reverse=True)
        target_pt = candidates[0]

if target_pt and os.path.exists(target_pt):
    file_size_mb = os.path.getsize(target_pt) / (1024 * 1024)
    print(f"📥 Tìm thấy file trọng số: {target_pt} ({file_size_mb:.2f} MB)")
    print("🚀 Đang mở tải file về máy tính của bạn...")
    files.download(target_pt)
    print("\n💡 Hướng dẫn tiếp theo sau khi tải về máy:")
    print("1. Đặt file vừa tải vào thư mục 'weights/best.pt' trong dự án IOT trên máy tính.")
    print("2. Mở PowerShell tại d:\\IOT và chạy lệnh: .\\run_video.ps1 -View để xem video trực tiếp!")
else:
    print("❌ Không tìm thấy file best.pt. Vui lòng kiểm tra lại quá trình huấn luyện ở Bước 5!")